In [5]:
import pandas as pd

In [6]:
# MBC 지반침하 고위험지역 주소
sinkhole_high_risk = pd.read_csv('./data/processed/지반침하_고위험지역_주소.csv')
# 서울시 건출묵 유출지하수
building_underground_water = pd.read_csv('./data/processed/building_underground_water_processed.csv')
# 서울시 건설 알림이 정보
construction = pd.read_csv('./data/processed/construction_processed.csv')
# 서울시 공사현장 유출지하수
construction_underground_water = pd.read_csv('./data/processed/construction_underground_water_processed.csv')
# 서울시 도로굴착 공사
road_construction = pd.read_csv('./data/processed/road_construction_processed.csv')
# 지하철 깊이
subway_depth = pd.read_csv('./data/processed/subway_depth.csv')

In [7]:
# 이 파일에 피쳐들 저장예정
features = pd.read_csv('./data/processed/subway_depth.csv')

## 지하철역 기본 구조

In [10]:
# 지하철역 기본 구조
import numpy as np
from geopy.distance import geodesic

# 정거장 깊이
features['station_depth'] = subway_depth['지반고'] - subway_depth['레일면고']

# 지하 층 수
features['underground_floors'] = subway_depth['층수'].str.extract('지하(\d+)')
features['underground_floors'] = pd.to_numeric(features['underground_floors'], errors='coerce').fillna(1)

# 플랫폼 형식별 복잡도
features['is_island_platform'] = (subway_depth['형식'] == '섬식').astype(int)
features['is_side_platform'] = (subway_depth['형식'] == '상대식').astype(int)

# 환승역 여부
transfer_keywords = ['환승', '분기', '연결']
features['is_transfer_station'] = subway_depth['비고'].str.contains('|'.join(transfer_keywords), na=False).astype(int)

# 호선별 특성 (건설 시기와 기술 수준)
features['line_1to4'] = subway_depth['호선'].isin([1,2,3,4]).astype(int)
features['line_5to8'] = subway_depth['호선'].isin([5,6,7,8]).astype(int)

<>:9: SyntaxWarning: invalid escape sequence '\d'
<>:9: SyntaxWarning: invalid escape sequence '\d'
/var/folders/mh/1w84fr7s5kxcwc2l24qrjjwc0000gn/T/ipykernel_11916/725895746.py:9: SyntaxWarning: invalid escape sequence '\d'
  features['underground_floors'] = subway_depth['층수'].str.extract('지하(\d+)')


In [11]:
features.head()

,연번,호선,역명,층수,형식,지반고,레일면고,선로기준정거장깊이,정거장깊이,비고,위도,경도,station_depth,underground_floors,is_island_platform,is_side_platform,is_transfer_station,line_1to4,line_5to8
0,1,1,서울,B2,섬식,129.99,117.04,12.95,11.85,"4호선,경의중앙선,공항철도환승",37.553150,126.972533,12.95,1.0,1,0,1,1,0
1,2,1,시청,B2,상대식,129.97,118.82,11.15,10.05,2호선환승,37.563590,126.975407,11.15,1.0,0,1,1,1,0
2,3,1,종각,B2,상대식,128.77,116.24,12.53,11.43,NaN,37.570203,126.983116,12.53,1.0,0,1,0,1,0
3,4,1,종로3가,B2,상대식,124.38,112.04,12.34,11.24,"3,5호선환승",37.570429,126.992095,12.34,1.0,0,1,1,1,0
4,5,1,종로5가,B2,상대식,121.75,109.26,12.49,11.39,NaN,37.570971,127.001900,12.49,1.0,0,1,0,1,0


## 지반침하 위험도

In [13]:
feats = []

for _, station in subway_depth.iterrows():
  station_coords = (station['위도'], station['경도'])
  
  # 거리 계산
  distances = []
  for _, risk_area in sinkhole_high_risk.iterrows():
    if pd.notna(risk_area['lat1']) and pd.notna(risk_area['lon1']):
      risk_coords = (risk_area['lat1'], risk_area['lon1'])
      distance = geodesic(station_coords, risk_coords).meters
      distances.append(distance)
      
  if distances:
    # 반경별 위험지역 개수
    risk_500m = sum(1 for d in distances if d <= 500)
    risk_1km = sum(1 for d in distances if d <= 1000)
    
    # 가장 가까운거
    nearest_distance = min(distances)
    
    # 위험도 가중 (거리 역)
    weighted_score = sum(1000/d for d in distances if d <= 500 and d > 0)
  else:
    risk_500m = risk_1km = 0
    nearest_distance = float('inf')
    weighted_score = 0
    
  feats.append({
    'station_name': station['역명'],
    'risk_area_count_500m': risk_500m,
    'risk_area_count_1km': risk_1km,
    'nearest_risk_distance': nearest_distance,
    'risk_weighted_score_500m': weighted_score
  })

In [16]:
feats = pd.DataFrame(feats)
features.rename(columns={'역명':'station_name'}, inplace=True)
features = features.merge(feats, on='station_name', how='left')

In [17]:
features.head()

,연번,호선,station_name,층수,형식,지반고,레일면고,선로기준정거장깊이,정거장깊이,비고,...,underground_floors,is_island_platform,is_side_platform,is_transfer_station,line_1to4,line_5to8,risk_area_count_500m,risk_area_count_1km,nearest_risk_distance,risk_weighted_score_500m
0,1,1,서울,B2,섬식,129.99,117.04,12.95,11.85,"4호선,경의중앙선,공항철도환승",...,1.0,1,0,1,1,0,0,0,2392.534837,0.000000
1,1,1,서울,B2,섬식,129.99,117.04,12.95,11.85,"4호선,경의중앙선,공항철도환승",...,1.0,1,0,1,1,0,0,0,2376.805396,0.000000
2,2,1,시청,B2,상대식,129.97,118.82,11.15,10.05,2호선환승,...,1.0,0,1,1,1,0,0,0,1322.292309,0.000000
3,2,1,시청,B2,상대식,129.97,118.82,11.15,10.05,2호선환승,...,1.0,0,1,1,1,0,0,0,1335.348492,0.000000
4,3,1,종각,B2,상대식,128.77,116.24,12.53,11.43,NaN,...,1.0,0,1,0,1,0,4,4,333.702328,10.042764
